# Expression Statement: timeout

Promela doesnt have **real-time features** - it is optimised for logic verification.

**Timeout** in Promela is abstraction allowing to model time-dependant logic (like in network protocols)



**timeout** - special variable in Promela 
- it becomes executable only if all other statements in the WHOLE SYSTEM are not executable
- provides escape from deadlock states
- read-only
- can not model time intervals, used to model only possible, not detailed real-time behavior



```python
active proctype watchdog(){
    do
    :: timeout -> guard!reset
    od
}

```

qulitative properties of the model
no quantitatives like probabilities/time bound and etc

tho we can simulate DISCRETE time, but we few BUT

# Simulating Time in Promela: 
Instead of using abstractions of timeouts, its also possible to *simulate (discrete) time*

```python
byte time; 

proctype Tick() {
    do
    :: timeout -> (time = (time + 1) % MAXTIME);
    od
}
```

Now other processes can wait for time to pass: 
```python
byte stamp;

do
:: atomic { toR ! MSG(data, sendb) -> stamp = time };
   if
   :: toS ? ACK(recvb) -> /* Good: received in time */
   :: time >= (stamp + SENDER_TIMEOUT) % MAXTIME -> /* Bad: timeout occurred */
   fi
od
```

This approach to simulate time works, but has its own disadvantages: 
1. all actions that take time have to synchronize on *time*
2. Expensive (due to Tick, #states can blow up)

Different variations of implementation of time exist.

NOTE: 1. since threy all become dependent on time, we essentially can not perform POR effectivly since all processes rely on each shared variable


# Control Flow Specifier: goto

**goto** - unconditional jump to a labeled statement
* each Promela statement can be labelled
* normally not executed, but used by a parser to determine target control state (point program counter to specific location)
* local to the proctype they are defined in. Must have unique name within its proctype




```python

L1: if 
    :: a != b -> goto L1
    :: a == b -> goto L2
    fi;
L2: ....

```

NOTE: eseentially goto creates transition to state with state that has diff program counter for that process that executed goto

# Exception Handling: unless

```promela
{ <stats> } unless {guard, <stats>}

```

- statements in stats are executed until guard becomes true
- unless construct is not a statement(it doesnt get executed), but a method to define structure of underlying automaton and distinguish between low and high priority transitions within a single process (so after guard becomes true, second part with priority gets executed).
- can reach inside atomic clauses 

# Macros in Promela
Promela uses C preprocessor *cpp* to handle macros, constants and additional compilation.
- All *cpp* commands start with #
- Promela supports embedding of C code using *c_block* syntax (from Version 4). However Promela cant see inside this block and treats it as a blackbox.  

- constants:   
```C
    #define MAX 4
```
   
- macros:  
```C
    #define RESET_ARRAY(a) \  
        d_step { a[0] = 0; a[1] = 1, a[2] = 2; }
```

- conditional fragments:
```C
    #define LOSSY 1  
    ....  
    #ifdef LOSSY  
    active proctype Daemon(){ }  
    #endif 
``` 

NOTE: read and replaces at the translation stage 

# Poor man's procedure's - inline
inline is a stylized version of a macro. (has style of a function, but no stack, no its own scope).  
- the body of the inline is directly pasted into the body of proctype at each point of invocation. 
- cant return a value to a caller.


```python
inline init_array(a){
    d_step {
        i = 0;
        do 
            :: i < N -> a[i] = 0; i++
            ........
        od
        i = 0;
    }
}
```

# (random) Simulation Algorithm

```Java
while (!error & !allBlocked) {
    // Visit all processes and collect all executable actions
    ActionList menu = getCurrentExecutableActions();
    
    // deadlock ≡ allBlocked
    allBlocked = (menu.size() == 0);
    
    if (!allBlocked) {
        // random simulation: act is chosen randomly by SPIN
        // interactive simulation: act is chosen by the user
        Action act = menu.chooseRandom();
        
        // act is executed and the system enters a new state
        error = act.execute();
    }
}
```

# Verification: overview


![title](images/P1.png)
    

NOTE: processes are perfromed concurrently: SPIN calculates all possible ways their actions can mix
For system Buchi, all states are accepting

For property Buchi (if any) accepting states are only ones satisfying Never Claiom

X should be empty to hold the property/ Searching for accepting states that are reachable from itself - implemented using 2 NDFS

So during verification SPIN takes 1 step from system buchi and asks if Never Claim has transition matching current system state. If yes- takes step, saves the paired state in hash table.
Otherwise trough it away

# Verification: DFS 
used by SPIN to generate and explore **complete state space** (happens on-the-fly)

```python
procedure dfs(s: state) {
    if error(s) {
        reportError(); // Found a safety violation!
    }
    
    foreach (successor t of s) {
        # 'Statespace' is the hash table
        if (t not in Statespace) {
            # Add to hash table and recurse
            Statespace.add(t);
            dfs(t);
        }
    }
}
```

# Verification of Properties 

## Safety property
- "nothing bad ever happens"   
- invariance - $x$ is always less then 5   
- deadlock freedom - system never reaches state where it is permanently stuck

What SPIN does? finds a trace leading to a "bad" thing. If there is no such trace --> property is satisfied

## Liveliness property
- "something good will eventually happen"
- termination - system will eventually terminate 
- response - if action X occur then action Y will eventually occur

What SPIN does? finds a (infinite) loop in which "good" thing does NOT happen. If there is no such loop --> property is satisfied

# Properties: liveliness and LTL

LTL formulae are used to specify liveliness properties
```C
    - []P    always P
    - <>P    eventually P
    - P U Q  P is true until Q becomes true
```
Some LTL patterns:

```C
    - invariance [] (p)
    - response   [] ((p) -> (<> (q)))
    - precedence [] ((p) -> ((q) U (r)))
    - objectvie  [] ((p) -> <>((q) || (r)))
```

We will explain how to run search for safety/liveliness properties in SPIN in a minute, but for now...

# Where to define LTL formula in Promela/SPIN? 

<mark> Option 1: </mark>
since SPIN version 6, we can write LTL directly inside ``` .pml ``` using ``` ltl ``` keyword.   
must define logical propositions first.


```C
/* 3. Define the LTL property at the bottom */
#define p (request_sent == true)
#define q (response_received == true)

ltl my_liveness_prop { [] (p -> <> q) }
```


then run specific property by name: 
```./pan -a -N my_liveness_prop ```       

NOTE: modern SPIN version is 6.5.2

<mark> Option 2: </mark>  
in command line:  
```spin -f "..." -a model.pml```

# How does SPIN represent states?: State Vectors

A **State Vector** is the information required to uniquely identify a system state. It contains:

* **Global variables**
* **Channel contents**
* **Process-specific data** (for each process):
    * Local variables
    * Process counter (PC)


## It is important to minimise the size of the state vector!!!

if **state vector** is m bytes and model has n **unique states** -> we will need mxn bytes to store full state space


# How SPIN stores the states?: default method

- hash table stores adresses of linked-lists of states
- states are stored explicitly
- fast lookup due to the hash function
- memory required:   
    **nxm** bytes + hash table

![title](images/hash.png)


# How SPIN stores the states?: bit-state hashing reduction technique

- hash table holds only bits: 0 or 1
- if hash table value is 1 for given index -> state already visited
- states are NOT stored explicitly
- fast look up due to the hash function
- memory required: only hash table

![title](images/bit_hash.png)


# SPIN verification report example
Lets see output of verification to see data about #states, state vector size and etc!

# SPIN Reduction algorithms
- SPIN has several optimisation algorithms to make verfication runs more effective: 
   - **POR**
   - **bitstate hashing**
   - minimised automaton encoding (BDD or minimised FSA)
   - state vector compression (finds reoccuring patterns and stores them as numberical references)
   - dataflow analysis (SPIN analyses Promela code and looks for "dead variables" and removes them)
   - slicing (pre-processing step: if we check property that only involves variables A and B, SPIN removes code related to other variables and that doesnt affect A and B)

SPIN supports several command-line options to select and further tune these optimization algorithms. 
Here are examples:

- <span style="color: red;">-DCOLLAPSE</span>: turn on state-vector compression
- <span style="color: red;">-DMA=N</span>: minimised automaton
- <span style="color: red;">-DBITSTATE</span>: bitstate hashing
- <span style="color: red;">-DNOREDUCE</span>: no POR

On where and when we add these flags in a few slides...

# SPIN Command-lines: search for properties options
We have to choose whether we want to verify a liveness or safety property for a single search:

* **Safety property:** This is the default (searches for failed assertions, deadlocks, and illegal steps). 
* **Acceptance cycles (General Liveness):** This default is changed to search for acceptance cycles if the `-a` flag is used. 
* **Non-acceptance cycles (Starvation, Livelock):** This is done by compiling the `pan.c` source with the `-DNP` compile-time directive and adding the `-l` flag for run-time. 
    * *Remark:* To find non-acceptance cycles, we need to label "good" actions in our code with a `progress` label.

# SPIN code processing pipeline: 3 stages of SPIN

1. <mark>Translation</mark>:
    - takes your Promela code and translates it to C code file pan.c 
    - ```bash spin -a [filename].pml ```
    - ```bash spin -f "..." -a [filename].pml ``` (-f stands for formula, optional)
    - <span style="color: red;">-a</span> flag stands for Automaton and tells SPIN to generate Buchi automaton for Never Claim from our LTL property
    -if no <span style="color: red;">-a</span> is used, SPIN performs simulation using pre-built interpretor:
        -  we can use <span style="color: red;">-run</span> or <span style="color: red;">-i</span> for random or interactive simulations
    - if <span style="color: red;">-a</span> is used but no LTL is provided, its okay, SPIN wont create Buchi automaton


2. <mark>Compilation</mark>: 
    - compile C code into Binary Executable file
    - ```bash gcc -D[flag] -o pan pan.c ```
    - <span style="color: red;">-D</span> stands for define, used to define C preprocessor marco 
    - example: use <span style="color: red;">-DSAFETY</span> for safety properties, its strips away all cycle-detection, makes search faster
    - <span style="color: red;">-DNP</span> means Define Non Progress used to find no-progress loops.
    - we can add flags for optimisation techniques like <span style="color: red;">-DCOLLAPSE</span> to shrink state vector



3. <mark>Execution</mark>:
    - ```bash ./pan [options] ```
    - add <span style="color: red;">-a</span> for search for acceptance cycles
    - <span style="color: red;">-l</span> for non-progress cycles
    - -f for weak fairness to either <span style="color: red;">-a</span> or <span style="color: red;">-l</span> to ignore unfair starvation counterexamples (SPIN doesnt support strong fairness)


# SPIN reduction techniques results: BRP
BRP = Bounded Retransmission Protocol
![title](images/table.png)


# Lets summarize: basic recipe to check $M \models \phi$

1. <mark>Sanity check</mark> using **interactive** or **random** simulations


2. <mark>Partial check</mark>: use SPIN's bitstate hashing to quickly sweep over the state space

3. <mark>Exhaustive check</mark>: if fails due to memory restrictions use optimisation techniques

# Final words

## Modelling priorities: give up speed for memory
1. minimise #states, state vector, max search depth, verification time
2. Often we use more than one validation model: worst case 1 model for each property